# colony_agents

> Phase 18 (`specs/multiagent/phase18/`): numpy members for colony rules and faulty members on the clue board. Honest rule agents: `ConfirmAgent` (the `confirm` arm), `ColonyAgent` (a shown value counts only when confirmed or held; confirms and contests), `GuardAgent` (never opens; posts, confirms, contests), `ThresholdGuard` (guards when contested clues pile up). Faulty members (the first `nLiar` slots): `ClueLiar` (independent), `ColludingLiar` (reposts fellow liars' false values until two liars back them), `MistakenAgent` (misreads 30% of the clues it holds, with no intent). `faultClass` and `memberClass` build them from a cell's world.

In [ ]:
#| default_exp colony_agents

In [ ]:
#| hide
from nbdev.showdoc import *

## A rule agent with its own view of the board

In [ ]:
#| export
from functools import partial

from explRL.clue_agents import ClueBoard, ClueLiar, RuleAgent, candidates, known
from explRL.clue_colony import Q, contestedCount, leading, tally


class ViewAgent(RuleAgent):
    "RuleAgent's choice with this agent's own clues (`heldFor`) and known clues (`knownFor`); unchanged by default."
    def heldFor(self, b, k):
        return b.held(self.id, k)

    def knownFor(self, b):
        return known(self.id, b, self.channel)

    def _needed(self, b, kn):
        "The open subtasks this agent shares for: all with policy 'always', those a teammate has open with 'needed'."
        others = {o for a, o in enumerate(b.opened) if a != self.id}
        return [k for k in sorted(kn) if self.share != 'needed' or k in others]

    def _shareable(self, b, kn, t):
        if self.channel is None or self.share == 'never':
            return None
        done = {(s[2], s[3]) for s in b.shareLog}
        for k in self._needed(b, kn):
            for j, r in sorted(self.heldFor(b, k).items()):
                if (k, j) not in done:
                    return ('share', (k, j, r))
        return None

    def choose(self, b, t):
        kn, k = self.knownFor(b), b.opened[self.id]
        if k in b.open and len(c := candidates(b, k, kn[k])) == 1:
            b.noteSubmit(self.id, 1)
            return ('submit', c[0])
        if act := self._shareable(b, kn, t):
            return act
        live = [x for x in b.open if b.tasks[x].expire > t + 1] or list(b.open)
        best = max(len(kn[x]) for x in live)
        top = [x for x in live if len(kn[x]) == best]
        top = [x for x in top if b.tasks[x].value == max(b.tasks[y].value for y in top)]
        if k not in top:
            return ('open', int(top[int(self.rng.integers(len(top)))]))
        if self.rng.random() < self.guess:
            c = candidates(b, k, kn[k])
            b.noteSubmit(self.id, len(c))
            return ('submit', int(c[int(self.rng.integers(len(c)))]))
        return ('wait', None)

## Honest rule agents for the colony arms

In [ ]:
#| export
class ConfirmAgent(ViewAgent):
    "The numpy `confirm` arm: first shares a held clue whose latest shown value differs; else as RuleAgent."
    def _shareable(self, b, kn, t):
        if self.channel is not None and self.share != 'never' and isinstance(self.channel, ClueBoard):
            for k in self._needed(b, kn):
                posts = self.channel.posts.get(k, [])
                for j, r in sorted(self.heldFor(b, k).items()):
                    last = max((x for x in posts if x[0] == j), key=lambda x: x[4], default=None)
                    if last is not None and last[2] != r:
                        return ('share', (k, j, r))
        return super()._shareable(b, kn, t)


class ColonyAgent(ViewAgent):
    "Counts a shown value only if confirmed (q senders or more) or held; shares a held clue whose leading value is missing, different or unconfirmed (and not already its own)."
    def __init__(self, channel, rng, share='needed', guess=0.0, q=Q):
        super().__init__(channel, rng, share, guess)
        self.q = int(q)

    def knownFor(self, b):
        tal, out = tally(self.channel, b), {k: {} for k in b.open}
        for k in out:
            for j, vs in tal.get(k, {}).items():
                if len(vs[0][1]) >= self.q:
                    out[k][j] = vs[0][0]
            out[k].update(self.heldFor(b, k))
        return out

    def _need(self, tal, k, j, r):
        lead = leading(tal, k, j)
        return lead is None or lead[0] != r or (len(lead[1]) < self.q and self.id not in lead[1])

    def _shareable(self, b, kn, t, ks=None):
        if self.channel is None or self.share == 'never':
            return None
        tal = tally(self.channel, b)
        for k in self._needed(b, kn) if ks is None else ks:
            for j, r in sorted(self.heldFor(b, k).items()):
                if self._need(tal, k, j, r):
                    return ('share', (k, j, r))
        return None


class GuardAgent(ColonyAgent):
    "Never opens: each turn shares one held clue whose leading value is missing, wrong or unconfirmed (earliest-expiring subtask first), else waits."
    def choose(self, b, t):
        ks = sorted(b.open, key=lambda k: (b.tasks[k].expire, k))
        return self._shareable(b, None, t, ks) or ('wait', None)


class ThresholdGuard(ColonyAgent):
    "A ColonyAgent that acts as a GuardAgent on turns when at least `theta` clues on the board are contested."
    def __init__(self, channel, rng, share='needed', guess=0.0, q=Q, theta=2):
        super().__init__(channel, rng, share, guess, q)
        self.theta = int(theta)

    def choose(self, b, t):
        if contestedCount(tally(self.channel, b)) >= self.theta:
            return GuardAgent.choose(self, b, t)
        return super().choose(b, t)

## Faulty members

In [ ]:
#| export
class ColludingLiar(ClueLiar):
    """On a lying turn (every other turn, as ClueLiar), first reposts a fellow liar's false value on an open subtask that
    fewer than two liars have posted; else lies as ClueLiar. Liars know each other (they register on the channel); every
    post by a liar is false, so no ground truth is used."""
    def choose(self, b, t):
        ch = self.channel
        if isinstance(ch, ClueBoard):
            mates = ch.__dict__.setdefault('_liars', set())
            mates.add(self.id)
            if (self.turn + 1) % 2:
                seen = {}
                for k in b.open:
                    for j, _, r, a, _ in ch.posts.get(k, []):
                        if a in mates:
                            seen.setdefault((k, j, r), set()).add(a)
                for (k, j, r), s in sorted(seen.items()):
                    if len(s) < 2 and self.id not in s:
                        self.turn += 1
                        return ('share', (k, j, r))
        return super().choose(b, t)


class MistakenAgent(ViewAgent):
    """An honest-looking rule agent (share 'needed') that misreads each clue it holds with probability `p`: once per
    clue, a wrong residue uniform among the others, used in its posts, its known clues and its answers."""
    def __init__(self, channel, rng, share='needed', guess=0.0, p=0.3):
        super().__init__(channel, rng, share, guess)
        self.p, self.read = float(p), {}

    def heldFor(self, b, k):
        out = {}
        for j, r in sorted(b.held(self.id, k).items()):
            if (k, j) not in self.read:
                q = b.tasks[k].task.primes[j]
                self.read[(k, j)] = int((r + 1 + self.rng.integers(q - 1)) % q) if self.rng.random() < self.p else r
            out[j] = self.read[(k, j)]
        return out

    def knownFor(self, b):
        out = known(self.id, b, self.channel)
        for k in out:
            out[k].update(self.heldFor(b, k))
        return out

## Building members from a cell's world

In [ ]:
#| export
def faultClass(world):
    "Factory(channel, rng) for scripted faulty slots: 'independent' (default) ClueLiar, 'collude' ColludingLiar, 'error' MistakenAgent."
    f = world.get('fault', 'independent')
    if f == 'adaptive':
        raise ValueError("fault 'adaptive' is played by Claude agents, not scripted members")
    return dict(independent=ClueLiar, collude=ColludingLiar, error=MistakenAgent)[f]


def scriptedFaults(world, nLiar):
    "rollout's `scripted` slots: {slot: factory} for scripted faults, None for none or for adaptive liars (the policy plays them)."
    return None if not nLiar or world.get('fault') == 'adaptive' else {i: faultClass(world) for i in range(nLiar)}


def faultMembers(world, nLiar, channel, rngs, llm):
    "The faulty slots' members: llm() for each when the fault is 'adaptive' (Stage 3), else faultClass members on rngs."
    return [llm() for _ in range(nLiar)] if world.get('fault') == 'adaptive' else [faultClass(world)(channel, r) for r in rngs]


def memberClass(world, slot, nLiar=0):
    """Factory(channel, rng) for an honest numpy slot: a guard slot (the first `guards` after the faulty ones) GuardAgent;
    'theta' ThresholdGuard; 'colony' ColonyAgent; 'confirm' ConfirmAgent; else RuleAgent (Phase 11, unchanged)."""
    share, guess, q = world.get('share', 'needed'), world.get('guess', 0.0), world.get('q', Q)
    if nLiar <= slot < nLiar + world.get('guards', 0):
        return partial(GuardAgent, share=share, guess=guess, q=q)
    if 'theta' in world:
        return partial(ThresholdGuard, share=share, guess=guess, q=q, theta=world['theta'])
    if world.get('colony'):
        return partial(ColonyAgent, share=share, guess=guess, q=q)
    if world.get('confirm'):
        return partial(ConfirmAgent, share=share, guess=guess)
    return partial(RuleAgent, share=share, guess=guess)

## Tests

In [ ]:
import numpy as np
from explRL.colony_agents import *
from explRL.clueboard import ClueBoardEnv
from explRL.clue_agents import ClueBoard, ClueLiar, RuleAgent
from explRL.clue_colony import tally

def board(seed=0, N=4):
    b = ClueBoardEnv(seed, N, m=2, r=2, R=30)
    return b, ClueBoard()

def member(cls, ch, i, **kw):
    m = cls(ch, np.random.default_rng(i), **kw)
    m.id = i
    return m

# ViewAgent at its defaults chooses as RuleAgent
for seed in range(5):
    b, ch = board(seed)
    a, v = member(RuleAgent, ch, 1), member(ViewAgent, ch, 1)
    assert a.choose(b, 0.0) == v.choose(b, 0.0)

# ColonyAgent ignores an unconfirmed false value and posts its held clue against it
b, ch = board(1)
i = 2
k, j = next((k, j) for k in b.open for j in b.held(i, k))
p, r = b.tasks[k].task.clue(j)
ch.post(0, k, (j, p, (r + 1) % p), 1.0)                          # one false post by agent 0
b.opened[1] = k                                                  # a teammate works on k: the clue is needed
c = member(ColonyAgent, ch, i)
assert c.knownFor(b)[k][j] == r                                  # own value, the false one not counted
assert c._shareable(b, c.knownFor(b), 2.0) == ('share', (k, j, r))
assert member(ColonyAgent, ch, 3).knownFor(b)[k].get(j) in (None, r)   # an unconfirmed false value is never known
ch.post(i, k, (j, p, r), 2.0)
assert c._shareable(b, c.knownFor(b), 3.0) != ('share', (k, j, r))     # already its own: not again
# ConfirmAgent posts against a differing latest value
b2, ch2 = board(1)
ch2.post(0, k, (j, p, (r + 1) % p), 1.0)
b2.opened[1] = k
cf = member(ConfirmAgent, ch2, i)
assert cf._shareable(b2, cf.knownFor(b2), 2.0) == ('share', (k, j, r))
# GuardAgent never opens; ThresholdGuard switches at theta
for seed in range(10):
    b3, ch3 = board(seed, 8)
    g = member(GuardAgent, ch3, 4)
    assert all(g.choose(b3, float(t))[0] in ('share', 'wait') for t in range(3))
b4, ch4 = board(2, 8)
tg = member(ThresholdGuard, ch4, 5, theta=1)
base = member(ColonyAgent, ch4, 5)
assert tg.choose(b4, 0.0) == base.choose(b4, 0.0)               # nothing contested: a forager
kk = b4.open[0]
pp, rr = b4.tasks[kk].task.clue(0)
ch4.post(0, kk, (0, pp, rr), 1.0)
ch4.post(1, kk, (0, pp, (rr + 1) % pp), 1.0)
assert tg.choose(b4, 1.0)[0] in ('share', 'wait')               # one contested clue: a guard (never opens)

# ColludingLiar reposts a fellow liar's false value within its next two lying turns
b5, ch5 = board(3, 8)
l0, l1 = member(ColludingLiar, ch5, 0), member(ColludingLiar, ch5, 1)
a0 = l0.choose(b5, 0.0)                                          # liar 0 lies first (turn 1 is a lying turn)
assert a0[0] == 'share'
k0, j0, r0 = a0[1]
ch5.post(0, k0, (j0, b5.tasks[k0].task.primes[j0], r0), 0.0)
acts = []
for t in range(4):
    act = l1.choose(b5, float(t))
    acts.append(act)
assert ('share', (k0, j0, r0)) in acts[:3]

# MistakenAgent: about 30% of held clues misread, one value per clue, used in its answers
wrong, total = 0, 0
for seed in range(200):
    b6, ch6 = board(seed, 4)
    m = MistakenAgent(ch6, np.random.default_rng(seed))
    m.id = 1
    for k in b6.open:
        got, truth = m.heldFor(b6, k), b6.held(1, k)
        assert got == m.heldFor(b6, k)                          # kept consistently
        wrong += sum(got[j] != truth[j] for j in truth)
        total += len(truth)
        assert all(m.knownFor(b6)[k][j] == got[j] for j in got)
assert 0.25 < wrong / total < 0.35, wrong / total

# factories from a world
w = dict(colony=True, guards=2)
assert faultClass({}) is ClueLiar and faultClass(dict(fault='collude')) is ColludingLiar and faultClass(dict(fault='error')) is MistakenAgent
try:
    faultClass(dict(fault='adaptive'))
    raise AssertionError('numpy cannot play adaptive liars')
except ValueError:
    pass
assert scriptedFaults({}, 0) is None and scriptedFaults(dict(fault='adaptive'), 2) is None and set(scriptedFaults({}, 2)) == {0, 1}
assert faultMembers(dict(fault='adaptive'), 2, None, [], lambda: 'llm') == ['llm', 'llm']
assert memberClass(w, 2, 2).func is GuardAgent and memberClass(w, 3, 2).func is GuardAgent and memberClass(w, 4, 2).func is ColonyAgent
assert memberClass({}, 0).func is RuleAgent and memberClass(dict(confirm=True), 0).func is ConfirmAgent
assert memberClass(dict(colony=True, theta=1), 3, 2).func is ThresholdGuard

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()